# Choosing a condition

Every AlphaGenome 2 prediction is made for one **condition**: the biological
context (such as a tissue, cell type, cell line, or your own population of
cells) that the model makes predictions for. Choosing the condition is the first step of
an analysis.

AlphaGenome 2 supports two ways to specify a condition:

| Mode | Arguments | When to use |
| :-- | :-- | :-- |
| **Reference condition** | `condition='UBERON:0000178_CL:0001054'` | Your tissue, cell type, or cell line is in `supported_conditions` and you want observed-track metadata. Covered in this tutorial. |
| **Expression conditioning** | `cell_conditioning=cells, cell_condition_name='...'` | You have human scRNA-seq raw UMI counts (`AnnData`, up to 256 cells per call via `dna_client.MAX_CONDITIONING_CELLS`). See [Conditioning on your own single-cell data](https://www.alphagenomedocs.com/colabs/v2/conditioning_on_single_cell_data.html). |

Key activities:

1.  List the supported reference conditions and search them by name.
2.  Check which output types are observed or imputed for a condition.
3.  Inspect the observed training tracks for a condition.

## Set up

In [1]:
# @title Install AlphaGenome

# @markdown Run this cell to install AlphaGenome.
from IPython.display import clear_output
! pip install alphagenome
clear_output()

In [2]:
# @title Imports
from alphagenome import colab_utils
from alphagenome.models.v2 import dna_client
from alphagenome.models.v2 import model_condition
import pandas as pd

Store your API key in Colab Secrets for persistent access across sessions (see [Installation](https://www.alphagenomedocs.com/installation.html#add-api-key-to-secrets)).

In [3]:
dna_model = dna_client.create(colab_utils.get_api_key())

## List the supported conditions

`supported_conditions` returns one row per reference condition, with its
ontology CURIE (`condition`), a readable `biosample_name`, `biosample_type`,
and the imputation mode of each output type:

In [4]:
conditions = dna_model.supported_conditions(dna_client.Organism.HOMO_SAPIENS)

print(f'{len(conditions)} supported human conditions')
conditions.head()

2799 supported human conditions


,condition,biosample_type,biosample_name,atac,cage,dnase,rna_seq,chip_histone,chip_tf,splice_sites,splice_site_usage,splice_junctions,contact_maps,sc_atac,sc_gene_expression,methylation
0,CL:0000047,primary_cell,neural stem cell,imputed_supported,imputed_exploratory,observed,observed,observed,imputed_exploratory,observed,observed,observed,imputed_exploratory,imputed_supported,imputed_supported,imputed_supported
1,CL:0000100,in_vitro_differentiated_cells,motor neuron,observed,imputed_exploratory,imputed_supported,observed,observed,imputed_exploratory,observed,observed,observed,imputed_exploratory,imputed_supported,imputed_supported,imputed_supported
2,CL:0000115,primary_cell,endothelial cell,imputed_supported,imputed_exploratory,observed,observed,imputed_supported,observed,imputed_exploratory,imputed_exploratory,imputed_exploratory,imputed_exploratory,imputed_supported,imputed_supported,imputed_supported
3,CL:0000134,primary_cell,mesenchymal stem cell,imputed_supported,imputed_exploratory,observed,observed,observed,imputed_exploratory,observed,observed,observed,imputed_exploratory,imputed_supported,imputed_supported,imputed_supported
4,CL:0000136,primary_cell,adipocyte,imputed_supported,imputed_exploratory,observed,imputed_supported,observed,imputed_exploratory,imputed_exploratory,imputed_exploratory,imputed_exploratory,imputed_exploratory,imputed_supported,imputed_supported,imputed_supported


## Search for a condition by name

Search the `biosample_name` column for the tissue or cell type you are
interested in. Here we look for CD14-positive monocytes:

In [5]:
monocytes = conditions[
    conditions['biosample_name'].str.contains('CD14-positive monocyte')
]
monocytes[['condition', 'biosample_name', 'rna_seq', 'atac']]

,condition,biosample_name,rna_seq,atac
442,UBERON:0000178_CL:0001054,blood CD14-positive monocyte,observed,imputed_supported
1451,UBERON:0002048_CL:0001054,lung CD14-positive monocyte,imputed_supported,imputed_supported
1515,UBERON:0002078_CL:0001054,right cardiac atrium CD14-positive monocyte,imputed_supported,imputed_supported
1544,UBERON:0002079_CL:0001054,left cardiac atrium CD14-positive monocyte,imputed_supported,imputed_supported
1571,UBERON:0002080_CL:0001054,heart right ventricle CD14-positive monocyte,imputed_supported,imputed_supported
1604,UBERON:0002084_CL:0001054,heart left ventricle CD14-positive monocyte,imputed_supported,imputed_supported
1636,UBERON:0002094_CL:0001054,interventricular septum CD14-positive monocyte,imputed_supported,imputed_supported
1734,UBERON:0002098_CL:0001054,apex of heart CD14-positive monocyte,imputed_supported,imputed_supported
2028,UBERON:0002185_CL:0001054,bronchus CD14-positive monocyte,imputed_supported,imputed_supported
2114,UBERON:0002351_CL:0001054,sinoatrial node CD14-positive monocyte,imputed_supported,imputed_supported


```{note}
Composite conditions join ontology CURIEs from `CLO`, `UBERON`, `CL`, `EFO`, and `NTR` with `_`, typically pairing a tissue `UBERON` CURIE with a cell-type `CL` CURIE such as `UBERON:0000178_CL:0001054` (CD14-positive monocytes in blood) or `UBERON:0002371_CL:0001054` (CD14-positive monocytes in bone marrow), while most bulk conditions use a single CURIE (such as `UBERON:0002048` for lung or an `EFO` cell line ID).
```

## Check which outputs are observed

For each output type, `supported_conditions` reports whether predictions for
that condition are `observed`, `imputed_supported`, or `imputed_exploratory`.
See [Navigating imputed tracks](https://www.alphagenomedocs.com/user_guides/imputation.html)
for what each mode means and pitfalls to keep in mind when using imputed tracks.

Let's look at which outputs are available for CD14-positive monocytes from blood:

In [6]:
condition = 'UBERON:0000178_CL:0001054'  # CD14-positive monocytes in blood.

conditions.set_index('condition').loc[[condition]].T

condition,UBERON:0000178_CL:0001054
biosample_type,primary_cell
biosample_name,blood CD14-positive monocyte
atac,imputed_supported
cage,imputed_exploratory
dnase,observed
rna_seq,observed
chip_histone,observed
chip_tf,observed
splice_sites,observed
splice_site_usage,observed


## Inspect the observed tracks

`observed_metadata` lists every track the model was trained on. Filtering it
to our condition shows the experiments behind the `observed` output types:

In [7]:
pd.set_option('display.max_colwidth', None)
observed = dna_model.observed_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()

observed[observed['biosample_curie'] == condition][
    ['output_type', 'Assay title', 'strand', 'name']
]

,output_type,Assay title,strand,name
190,OutputType.DNASE,DNase-seq,.,UBERON:0000178_CL:0001054_DNase-seq-encode-paired-ended
191,OutputType.DNASE,DNase-seq,.,UBERON:0000178_CL:0001054_DNase-seq-encode-single-ended
450,OutputType.RNA_SEQ,polyA plus RNA-seq,+,UBERON:0000178_CL:0001054_polyA plus RNA-seq-encode-paired-ended
451,OutputType.RNA_SEQ,polyA plus RNA-seq,-,UBERON:0000178_CL:0001054_polyA plus RNA-seq-encode-paired-ended
452,OutputType.RNA_SEQ,polyA plus RNA-seq,.,UBERON:0000178_CL:0001054_polyA plus RNA-seq-encode-paired-ended
453,OutputType.RNA_SEQ,polyA plus RNA-seq,.,UBERON:0000178_CL:0001054_polyA plus RNA-seq-gtex-paired-ended
599,OutputType.CHIP_HISTONE,Histone ChIP-seq,.,UBERON:0000178_CL:0001054_Histone ChIP-seq-encode-H2AFZ-single-ended
600,OutputType.CHIP_HISTONE,Histone ChIP-seq,.,UBERON:0000178_CL:0001054_Histone ChIP-seq-encode-H3K27ac-single-ended
601,OutputType.CHIP_HISTONE,Histone ChIP-seq,.,UBERON:0000178_CL:0001054_Histone ChIP-seq-encode-H3K27me3-single-ended
602,OutputType.CHIP_HISTONE,Histone ChIP-seq,.,UBERON:0000178_CL:0001054_Histone ChIP-seq-encode-H3K36me3-single-ended


The same metadata summarizes training coverage across all conditions, for
example the number of observed tracks per output type in human and mouse (see
[Model output metadata](https://www.alphagenomedocs.com/exploring_model_metadata.html)):

In [8]:
human_tracks = (
    observed.groupby('output_type').size().rename('# Human tracks')
)
mouse_tracks = (
    dna_model.observed_metadata(dna_client.Organism.MUS_MUSCULUS)
    .concatenate()
    .groupby('output_type')
    .size()
    .rename('# Mouse tracks')
)

pd.concat([human_tracks, mouse_tracks], axis=1).astype(pd.Int64Dtype())

,# Human tracks,# Mouse tracks
output_type,,
OutputType.ATAC,179,27
OutputType.CAGE,550,188
OutputType.DNASE,385,86
OutputType.RNA_SEQ,738,206
OutputType.CHIP_HISTONE,1183,190
OutputType.CHIP_TF,1709,145
OutputType.SPLICE_SITES,1135,295
OutputType.SPLICE_SITE_USAGE,560,126
OutputType.SPLICE_JUNCTIONS,280,63


## Using the condition

Pass the ontology CURIE string directly as `condition=` or construct a
`model_condition.ModelCondition` from component CURIEs:

In [9]:
model_condition.from_curies(['UBERON:0000178', 'CL:0001054'])

ModelCondition(clo=None, uberon=OntologyTerm(type=<OntologyType.UBERON: 2>, id=178), cl=OntologyTerm(type=<OntologyType.CL: 3>, id=1054), efo=None, ntr=None)

`predict_interval`, `predict_variant`, and `score_variant` accept one condition
per call (and multiple requested output types concurrently); to compare several
conditions, loop over them or pass matching lists to `predict_intervals`,
`predict_variants`, or `score_variants`.

### Using your own single-cell data instead

To use expression conditioning (`cell_conditioning=` alongside
`cell_condition_name=`) with your own human scRNA-seq counts instead of a
reference `condition=`, request
`dna_client.OutputType.SC_GENE_EXPRESSION_DENOISED` for denoised single-cell
gene expression (one channel per strand, in CPM); see
[Conditioning on your own single-cell data](https://www.alphagenomedocs.com/colabs/v2/conditioning_on_single_cell_data.html).

Next in the core workflow, we will
[select the genomic region](https://www.alphagenomedocs.com/colabs/v2/selecting_a_genomic_region.html)
to make predictions for.